# Toy notebook

This is a quick notebook to test the Julia CGE project of to compute JS-divergence scores for graph embeddings.

The Julia language should be installed: https://julialang.org/downloads/

CGE project should be installed, see: https://github.com/KrainskiL/CGE.jl 



In [1]:
import igraph as ig
import numpy as np
import pickle
import scipy.sparse.linalg as lg
import subprocess

## make sure the paths are correct
datadir = '../Datasets/' ## the book's datasets
julia = 'julia' ## the Julia executable
CGE = '../CGE/CGE_CLI.jl ' ## the CGE script


In [2]:
## Laplacian eigenmaps embedding
def LE(g, dim=2):
    L_sym = np.array(g.laplacian(normalized=True))
    w, v = lg.eigs(L_sym, k=dim + 1, which='SM')
    idx = np.argsort(w) # sort eigenvalues
    w = w[idx]
    v = v[:, idx]
    X = v[:, 1:]
    return X.real

## save an embedding to disk with default filename "_embed" (or some other user-supplied name); 
## this file will be used to compute divergence with Julia CGE framework
def saveEmbedding(X, g, fn='_embed'):
    with open(fn,'w') as f:
        f.write(str(X.shape[0]) + " " + str(X.shape[1])+'\n')
        for i in range(X.shape[0]):
            f.write(g.vs[i]['name']+' ')
            for j in range(X.shape[1]):
                f.write(str(X[i][j])+' ')
            f.write('\n')

## Computing Jensen-Shannon (JS) divergence with the Julia CGE framework code 
## required input files: edgelist, communities and embedding
def JS(edge_file, comm_file, embed_file, return_local=True, seed=123):
    cmd = julia+' '+CGE+' -g '+edge_file+' -c '+comm_file+' -e '+embed_file+' --seed '+str(seed)+' 2>_stderr' 
    print('running the following command:',cmd)
    s = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE)
    x = s.stdout.decode().split(',')
    if len(x[0]) == 0:
        print('Error running CLI command:\n', cmd )
        print('Make sure Julia and CGE are correctly installed')
        return -1
    if return_local:
        return (float(x[1]),float(x[5]))
    else:
        return float(x[1])


In [3]:
## this will create the "_embed" file
edgefile=datadir+'Zachary/zachary.edgelist'
commfile=datadir+'Zachary/zachary.ecg'
zac = ig.Graph.Read_Ncol(edgefile,directed=False)

## compute some 2-d embedding and save the file
X = LE(zac, dim=2) 
saveEmbedding(X, zac, fn='./_embed')

## this calls the CGE.jl script to compute JS divergence
div = JS(edgefile, commfile, embed_file='./_embed')
print('divergence scores;',div)


running the following command: julia ../CGE/CGE_CLI.jl  -g ../Datasets/Zachary/zachary.edgelist -c ../Datasets/Zachary/zachary.ecg -e ./_embed --seed 123 2>_stderr
divergence scores; (0.008543633514461194, 0.030399999999999983)


### embedding

below is an example of the format for the embedding file (here 2-d embedding for the zachary graph)
